# 05 - Evaluate LLaMEA Champions (N=10 Independent Runs)

This notebook:
1. Loads problem-specific **Clean** and **Noisy** champions from  (generated by Notebook 03).
2. Executes each champion **N=10 independent times** on target BBOB problems across multiple dimensions and noise levels.
3. Evaluates Clean Champions on clean settings () and Noisy Champions on noisy settings ().
4. Attaches IOH Analyzer context manager to output IOH performance files to .


In [ ]:
import sys
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
from pathlib import Path

# Add src to path
cwd = Path(".").resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core.config import DATA_DIR, PROJECT_ROOT
from domain.services.noise_strategy import MultiplicativeNoiseStrategy, NoNoiseStrategy
from infra.problems.bbob import BBOBProblem
from infra.storage import get_db_connection
from synthesis.execution import AlgorithmExecutor

# ── User Execution Controls & Selective Filters ──────────────────────────────
FORCE_REEVALUATE  = False   # Set True to bypass cache and re-evaluate all
FILTER_PROBLEMS   = None    # e.g., [8, 11] to evaluate specific problems, or None for all
FILTER_STRATEGIES = None    # e.g., ["baseline", "guided"] or None for all
FILTER_MODES      = None    # e.g., ["clean"], ["noisy"], or None for all
FILTER_DIMS       = None    # e.g., [2, 3], or None (uses all DIMS in database)
N_RUNS            = 10      # Number of independent benchmark runs per config
TIMEOUT_SECONDS   = 30.0    # Per-run execution timeout in seconds

# ── Dynamic Experiment Parameters Extracted Purely from Database ──────────────
CHAMPIONS_PATH = DATA_DIR / "champions.json"
IOH_LOGS_DIR   = DATA_DIR / "ioh_logs"

with get_db_connection() as conn:
    df_exp_meta = pd.read_sql_query(
        "SELECT DISTINCT dim, budget, noise_std, problem_id FROM experiments WHERE status = 'completed'",
        conn
    )

if df_exp_meta.empty:
    raise RuntimeError("No completed experiments found in database. Please run Notebook 02 first.")

DIMS   = sorted(df_exp_meta["dim"].dropna().astype(int).unique().tolist())
BUDGET = int(df_exp_meta["budget"].dropna().max())

# Apply FILTER_DIMS if specified
if FILTER_DIMS is not None:
    DIMS = [d for d in DIMS if d in FILTER_DIMS]

print(f"🎯 Dynamic parameters loaded from database:")
print(f"  • Evaluated Dimensions: {DIMS}")
print(f"  • Benchmark Budget:     {BUDGET}")
print(f"  • Runs per config:      {N_RUNS}")
print(f"  • Champions JSON:       {CHAMPIONS_PATH}")
print(f"  • IOH Logs Output:      {IOH_LOGS_DIR}")
print(f"  • Force Re-evaluate:    {FORCE_REEVALUATE}")
if FILTER_PROBLEMS:   print(f"  • Filter Problems:      f{FILTER_PROBLEMS}")
if FILTER_STRATEGIES: print(f"  • Filter Strategies:    {FILTER_STRATEGIES}")
if FILTER_MODES:      print(f"  • Filter Modes:         {FILTER_MODES}")


## 1. Load Champions JSON

In [5]:
if not CHAMPIONS_PATH.exists():
    raise FileNotFoundError(f"Champions file not found at {CHAMPIONS_PATH}. Please run Notebook 04 first.")

with open(CHAMPIONS_PATH, 'r', encoding='utf-8') as f:
    champions = json.load(f)

print(f"Loaded {len(champions)} champion configuration(s):")
for key, info in champions.items():
    mode_str = info.get('mode', 'all').upper()
    print(f"  {key} [{mode_str}]: {info['algorithm_name']} (from Exp #{info['experiment_id']}, std={info['noise_std']})")


Loaded 40 champion configuration(s):
  f1_clean_baseline [CLEAN]: ImprovedHillClimbing (from Exp #13, std=0.0)
  f1_noisy_baseline [NOISY]: ImprovedRandomSearchOptimizer (from Exp #2, std=0.05)
  f1_clean_guided [CLEAN]: DEOptimizer (from Exp #63, std=0.0)
  f1_noisy_guided [NOISY]: NoisyOptimizationAlgorithm (from Exp #82, std=0.05)
  f1_clean_thinking [CLEAN]: EnhancedOptimizer (from Exp #87, std=0.0)
  f1_noisy_thinking [NOISY]: EnhancedStochasticOptimizer (from Exp #142, std=0.05)
  f1_clean_vectorization [CLEAN]: EnhancedGradientDirectedSearch (from Exp #59, std=0.0)
  f1_noisy_vectorization [NOISY]: ImprovedNoisyOptimization (from Exp #107, std=0.05)
  f8_clean_baseline [CLEAN]: ImprovedDifferentialEvolution (from Exp #118, std=0.0)
  f8_noisy_baseline [NOISY]: ImprovedEvolutionaryAlgorithm (from Exp #16, std=0.05)
  f8_clean_guided [CLEAN]: EnhancedLandscapeExploiter (from Exp #133, std=0.0)
  f8_noisy_guided [NOISY]: EnhancedNoisyCMAES (from Exp #83, std=0.05)
  f8_clean_thinki

## 2. Execute Champion Evaluation Benchmark

In [ ]:
from infra.problems import ProblemAnalyzer
from synthesis.execution import AlgorithmExecutor

executor = AlgorithmExecutor(timeout_seconds=TIMEOUT_SECONDS)

skipped = []
evaluated = []

def _compute_code_hash(code_content: str) -> str:
    """Compute a SHA-256 hash of the champion code for cache invalidation."""
    return hashlib.sha256(code_content.encode("utf-8")).hexdigest()

def _read_provenance(prov_path: Path) -> dict | None:
    """Read provenance.json, returning None if missing or malformed."""
    try:
        return json.loads(prov_path.read_text(encoding="utf-8"))
    except Exception:
        return None

def _write_provenance(prov_path: Path, info: dict, dim: int, noise_std: float, code_hash: str, med_err: float) -> None:
    """Write provenance metadata alongside the IOH log folder."""
    prov = {
        "experiment_id":     int(info.get("experiment_id", -1)),
        "iteration_id":      int(info.get("iteration_id", -1)) if "iteration_id" in info else None,
        "algorithm_name":    info["algorithm_name"],
        "code_path":         info["code_path"],
        "code_hash":         code_hash,
        "problem_id":        int(info["problem_id"]),
        "dim":               dim,
        "noise_std":         noise_std,
        "prompt_strategy":   info.get("prompt_strategy", "baseline"),
        "mode":              info.get("mode", "all"),
        "n_runs":            N_RUNS,
        "median_clean_error": float(med_err) if not np.isinf(med_err) else None,
        "evaluated_at":      pd.Timestamp.now().isoformat(),
    }
    prov_path.write_text(json.dumps(prov, indent=2), encoding="utf-8")

for key, info in champions.items():
    p_id  = int(info["problem_id"])
    mode  = info.get("mode", "all").lower()
    strat = info.get("prompt_strategy", "baseline").lower()
    champ_noise_std = float(info.get("noise_std", 0.0))
    exp_id = int(info.get("experiment_id", -1))

    # ── User Filters ──────────────────────────────────────────────────────────
    if FILTER_PROBLEMS and p_id not in FILTER_PROBLEMS:
        continue
    if FILTER_STRATEGIES and strat not in FILTER_STRATEGIES:
        continue
    if FILTER_MODES and mode not in FILTER_MODES:
        continue

    code_file = (
        PROJECT_ROOT / info["code_path"]
        if not Path(info["code_path"]).is_absolute()
        else Path(info["code_path"])
    )
    if not code_file.exists():
        print(f"[WARN] Code file for {key} not found at {code_file}. Skipping.")
        continue

    code_content = code_file.read_text(encoding="utf-8")
    algo_name    = info["algorithm_name"]
    code_hash    = _compute_code_hash(code_content)

    eval_noise_levels = [0.0] if mode == "clean" else [champ_noise_std]

    for dim in DIMS:
        for noise_std in eval_noise_levels:
            out_dir           = IOH_LOGS_DIR / f"{dim}D" / f"std_{noise_std}" / f"f{p_id}"
            folder_name       = f"llamea_{strat}_{mode}"
            target_log_folder = out_dir / folder_name
            prov_path         = target_log_folder / "provenance.json"

            # ── Provenance Check (cache validation) ──────────────────────────
            if not FORCE_REEVALUATE and target_log_folder.exists():
                prov = _read_provenance(prov_path)
                logs_exist = (
                    any(target_log_folder.glob("*.json")) and
                    any(target_log_folder.glob("**/*.dat"))
                )
                if (
                    prov is not None
                    and logs_exist
                    and prov.get("experiment_id") == exp_id
                    and prov.get("code_hash")     == code_hash
                ):
                    msg = f"⏭️  [SKIP] f{p_id} {mode.upper()} [{strat}] ({dim}D, noise={noise_std}): Exp #{exp_id} already evaluated (Median err={prov.get('median_clean_error')})"
                    print(msg)
                    skipped.append(key)
                    continue
                elif logs_exist and (prov is None or prov.get("experiment_id") == exp_id):
                    # Existing valid logs: preserve and seed provenance
                    print(f"📦 [PRESERVED] f{p_id} {mode.upper()} [{strat}] ({dim}D, noise={noise_std}): Existing valid logs for Exp #{exp_id}. Generating provenance.")
                    _write_provenance(prov_path, info, dim, noise_std, code_hash, float('nan'))
                    skipped.append(key)
                    continue
                else:
                    old_exp = prov.get("experiment_id", "unknown") if prov else "unknown"
                    reason  = "new champion" if prov and prov.get("experiment_id") != exp_id else "code changed or logs missing"
                    print(f"🔄 [UPDATE] f{p_id} {mode.upper()} [{strat}] ({dim}D): {reason} — old Exp #{old_exp}, new Exp #{exp_id}")
                    # Safely remove ONLY the specific champion sub-folder
                    shutil.rmtree(target_log_folder, ignore_errors=True)
            elif FORCE_REEVALUATE and target_log_folder.exists():
                print(f"⚡ [FORCE] f{p_id} {mode.upper()} [{strat}] ({dim}D, noise={noise_std}): Forcing re-evaluation")
                # Safely remove ONLY the specific champion sub-folder
                shutil.rmtree(target_log_folder, ignore_errors=True)

            out_dir.mkdir(parents=True, exist_ok=True)

            print()
            print(f"🚀 [EVALUATE] f{p_id} {mode.upper()} [{strat}] ({dim}D, noise={noise_std}): {algo_name} | Exp #{exp_id} | N={N_RUNS} runs, Budget={BUDGET}")

            noise_strat = MultiplicativeNoiseStrategy(noise_std) if noise_std > 0.0 else NoNoiseStrategy()
            problem = BBOBProblem(
                problem_id=p_id,
                dim=dim,
                instance_id=1,
                noise_strategy=noise_strat,
            )

            clean_errors = []
            with ProblemAnalyzer(
                problem=problem,
                algorithm_name=f"LLaMEA {strat.capitalize()} Champion",
                folder_name=folder_name,
            ):
                for run_idx in range(1, N_RUNS + 1):
                    problem.reset()
                    try:
                        best_x, best_y = executor.execute_algorithm(
                            code=code_content,
                            name=algo_name,
                            dim=dim,
                            problem=problem.get_objective_fn(),
                            budget=BUDGET,
                        )
                        if best_x is not None:
                            clean_y   = problem.eval_clean(best_x)
                            clean_err = abs(clean_y - problem.true_optimum)
                        else:
                            clean_err = float("inf")
                        clean_errors.append(clean_err)
                        print(f"  Run {run_idx:2d}/{N_RUNS}: evals={problem.evaluations}, final clean error={clean_err:.6e}")
                    except Exception as exc:
                        print(f"  Run {run_idx:2d}/{N_RUNS} FAILED: {exc}")
                        clean_errors.append(float("inf"))

            valid_errs = [e for e in clean_errors if not np.isinf(e)]
            med_err    = np.median(valid_errs) if valid_errs else float("inf")
            print(f"  ✅ f{p_id} {mode.upper()} [{strat}] ({dim}D, noise={noise_std}) Median Clean Error: {med_err:.6e}")

            # ── Write provenance alongside logs ───────────────────────────────
            _write_provenance(prov_path, info, dim, noise_std, code_hash, med_err)

            evaluated.append(key)

print()
print(f"✨ Evaluation complete!")
print(f"   🚀 Evaluated : {len(evaluated)} configs")
print(f"   ⏭️  Skipped   : {len(skipped)} configs (already up-to-date)")
if skipped:
    print(f"   Skipped keys: {len(set(skipped))} unique champions")
